In [ ]:
import pandas as pd
import glob

# load all csv files
files = glob.glob("*.csv")
df_list = []

for file in files:
    df = pd.read_csv(file)

    # clean column names
    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        'Option type': 'Option Type',
        'Open Int': 'Open Interest'
    })

    # FIX: Add 'Symbol' (or whatever the column is named in your raw file) to this list
    df = df[['Symbol', 'Date', 'Expiry', 'Strike Price', 'Option Type',
             'Close', 'Open Interest', 'Underlying Value']]

    df_list.append(df)

# combine all
final_df = pd.concat(df_list, ignore_index=True)

# sort data
final_df = final_df.sort_values(by=['Symbol', 'Date', 'Expiry', 'Strike Price'])

# save to excel
final_df.to_excel("uncleaned_stockoptiondata.xlsx", index=False)
print("Done!")

Done!


In [ ]:
import pandas as pd

# load file
df = pd.read_excel("uncleaned_stockoptiondata.xlsx")

# clean column names
df.columns = df.columns.str.strip()

# convert columns
df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%Y', errors='coerce')
df['Expiry'] = pd.to_datetime(df['Expiry'], format='%d-%b-%Y', errors='coerce')

df['Open Interest'] = pd.to_numeric(df['Open Interest'], errors='coerce')
df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
df['Underlying Value'] = pd.to_numeric(df['Underlying Value'], errors='coerce')

# sort
df = df.sort_values(by=['Date', 'Expiry', 'Strike Price'])

# save
df.to_excel("stockoptiondata.xlsx", index=False)

print("Final cleaned file ready!")

Final cleaned file ready!


In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os

# ==========================================
# 1. Load and Prepare the Data
# ==========================================
nifty_file = 'NIFTYoptiondata.xlsx'
stock_file = 'stockoptiondata.xlsx'

df_nifty = pd.read_excel(nifty_file)
df_stock = pd.read_excel(stock_file)

def preprocess_data(df):
    """
    Converts date columns and calculates Maturity (in years).
    """
    df['Date'] = pd.to_datetime(df['Date'])
    df['Expiry'] = pd.to_datetime(df['Expiry'])

    # Calculate Maturity (T - t) in years. Assume 365 days in a year.
    df['Maturity'] = (df['Expiry'] - df['Date']).dt.days / 365.0

    # Drop expired options and rows with missing vital data
    df = df[df['Maturity'] > 0]
    df = df.dropna(subset=['Strike Price', 'Maturity', 'Close', 'Option Type'])
    return df

df_nifty = preprocess_data(df_nifty)
df_stock = preprocess_data(df_stock)

# Directory Setup Helper
BASE_PLOT_DIR = "Output_Plots"
def ensure_dir(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

# ==========================================
# 2. 3D Plotting Function
# ==========================================
def plot_3d_options(df, title_prefix):
    calls = df[df['Option Type'] == 'CE']
    puts = df[df['Option Type'] == 'PE']

    fig = plt.figure(figsize=(14, 6))

    # Call Options 3D Plot
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.scatter(calls['Strike Price'], calls['Maturity'], calls['Close'], c='blue', alpha=0.6)
    ax1.set_xlabel('Strike Price')
    ax1.set_ylabel('Maturity (Years)')
    ax1.set_zlabel('Option Price')
    ax1.set_title(f'{title_prefix} - Call Options (3D)')

    # Put Options 3D Plot
    ax2 = fig.add_subplot(122, projection='3d')
    ax2.scatter(puts['Strike Price'], puts['Maturity'], puts['Close'], c='red', alpha=0.6)
    ax2.set_xlabel('Strike Price')
    ax2.set_ylabel('Maturity (Years)')
    ax2.set_zlabel('Option Price')
    ax2.set_title(f'{title_prefix} - Put Options (3D)')

    plt.tight_layout()

    save_folder = os.path.join(BASE_PLOT_DIR, title_prefix)
    ensure_dir(save_folder)
    save_path = os.path.join(save_folder, f'{title_prefix}_3D_Plot.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close() # Close to prevent memory overload when looping

# ==========================================
# 3. 2D Plotting Function
# ==========================================
def plot_2d_observations(df, title_prefix):
    calls = df[df['Option Type'] == 'CE']
    puts = df[df['Option Type'] == 'PE']

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Price vs Strike
    axes[0, 0].scatter(calls['Strike Price'], calls['Close'], alpha=0.5, color='blue')
    axes[0, 0].set_title(f'{title_prefix} - Call Price vs Strike')
    axes[0, 0].set_xlabel('Strike Price')
    axes[0, 0].set_ylabel('Option Price')

    axes[0, 1].scatter(puts['Strike Price'], puts['Close'], alpha=0.5, color='red')
    axes[0, 1].set_title(f'{title_prefix} - Put Price vs Strike')
    axes[0, 1].set_xlabel('Strike Price')
    axes[0, 1].set_ylabel('Option Price')

    # Price vs Maturity
    axes[1, 0].scatter(calls['Maturity'], calls['Close'], alpha=0.5, color='blue')
    axes[1, 0].set_title(f'{title_prefix} - Call Price vs Maturity')
    axes[1, 0].set_xlabel('Maturity (Years)')
    axes[1, 0].set_ylabel('Option Price')

    axes[1, 1].scatter(puts['Maturity'], puts['Close'], alpha=0.5, color='red')
    axes[1, 1].set_title(f'{title_prefix} - Put Price vs Maturity')
    axes[1, 1].set_xlabel('Maturity (Years)')
    axes[1, 1].set_ylabel('Option Price')

    plt.tight_layout()

    save_folder = os.path.join(BASE_PLOT_DIR, title_prefix)
    ensure_dir(save_folder)
    save_path = os.path.join(save_folder, f'{title_prefix}_2D_Plot.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close() # Close figure

# ==========================================
# 4. Execute Plots for ALL Data
# ==========================================

print("Generating NIFTY plots...")
plot_3d_options(df_nifty, "NIFTY")
plot_2d_observations(df_nifty, "NIFTY")

# Dynamically loop through every single stock in the file (ITC, RELIANCE, INFY, etc.)
unique_stocks = df_stock['Symbol'].unique()
print(f"Found stock symbols: {unique_stocks}")

for symbol in unique_stocks:
    print(f"Generating plots for {symbol}...")
    df_symbol = df_stock[df_stock['Symbol'] == symbol]
    plot_3d_options(df_symbol, symbol)
    plot_2d_observations(df_symbol, symbol)

print(f"All plots successfully saved in the '{BASE_PLOT_DIR}' directory!")

Generating NIFTY plots...
Found stock symbols: ['ITC' 'HDFCBANK' 'INFY' 'RELIANCE']
Generating plots for ITC...
Generating plots for HDFCBANK...
Generating plots for INFY...
Generating plots for RELIANCE...
All plots successfully saved in the 'Output_Plots' directory!


In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import norm
import os

# ==========================================
# 1. BSM Formula and Newton-Raphson Method
# ==========================================

def bsm_price(S, K, T, r, sigma, option_type):
    """Calculates the BSM option price."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'CE':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == 'PE':
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def bsm_vega(S, K, T, r, sigma):
    """Calculates the Vega (derivative of price w.r.t volatility)."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return S * np.sqrt(T) * norm.pdf(d1)

def implied_volatility(market_price, S, K, T, r, option_type, tol=1e-5, max_iter=100):
    """Finds Implied Volatility using the Newton-Raphson method."""
    sigma = 0.5  # Initial guess (50% volatility)

    for i in range(max_iter):
        price = bsm_price(S, K, T, r, sigma, option_type)
        diff = price - market_price

        # If the difference is within our tolerance, we found the IV
        if abs(diff) < tol:
            return sigma

        vega = bsm_vega(S, K, T, r, sigma)

        # Avoid division by zero if vega gets too small
        if vega < 1e-8:
            break

        # Newton-Raphson step
        sigma = sigma - diff / vega

        # Volatility cannot be negative; reset to a small positive number if it drops below 0
        if sigma <= 0:
            sigma = 0.001

    # Return NaN if it doesn't converge
    return np.nan

# ==========================================
# 2. Load Data and Calculate IV
# ==========================================

nifty_file = 'NIFTYoptiondata.xlsx'
stock_file = 'stockoptiondata.xlsx'

df_nifty = pd.read_excel(nifty_file)
df_stock = pd.read_excel(stock_file)
r = 0.05  # Risk-free rate = 5%

def preprocess_and_calc_iv(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df['Expiry'] = pd.to_datetime(df['Expiry'])
    df['Maturity'] = (df['Expiry'] - df['Date']).dt.days / 365.0

    df = df[df['Maturity'] > 0].copy()
    df = df.dropna(subset=['Strike Price', 'Maturity', 'Close', 'Option Type', 'Underlying Value'])

    # Calculate IV row by row
    print("Calculating Implied Volatilities (this may take a moment)...")
    df['Implied_Vol'] = df.apply(lambda row: implied_volatility(
        market_price=row['Close'],
        S=row['Underlying Value'],
        K=row['Strike Price'],
        T=row['Maturity'],
        r=r,
        option_type=row['Option Type']
    ), axis=1)

    # Drop rows where Newton-Raphson failed to converge
    df = df.dropna(subset=['Implied_Vol'])

    # Filter out extreme outliers (e.g., IV > 300%) resulting from deep out-of-the-money illiquid options
    df = df[df['Implied_Vol'] < 3.0]
    return df

df_nifty = preprocess_and_calc_iv(df_nifty)
df_stock = preprocess_and_calc_iv(df_stock)

# Directory Setup
BASE_PLOT_DIR = "Output_Plots_Part_B"
if not os.path.exists(BASE_PLOT_DIR):
    os.makedirs(BASE_PLOT_DIR)

def ensure_dir(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

# ==========================================
# 3. Plotting Functions
# ==========================================

def plot_3d_iv(df, title_prefix):
    calls = df[df['Option Type'] == 'CE']
    puts = df[df['Option Type'] == 'PE']

    fig = plt.figure(figsize=(14, 6))

    # Call Options 3D IV Plot
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.scatter(calls['Strike Price'], calls['Maturity'], calls['Implied_Vol'], c='blue', alpha=0.6)
    ax1.set_xlabel('Strike Price')
    ax1.set_ylabel('Maturity (Years)')
    ax1.set_zlabel('Implied Volatility')
    ax1.set_title(f'{title_prefix} - Call Implied Volatility (3D)')

    # Put Options 3D IV Plot
    ax2 = fig.add_subplot(122, projection='3d')
    ax2.scatter(puts['Strike Price'], puts['Maturity'], puts['Implied_Vol'], c='red', alpha=0.6)
    ax2.set_xlabel('Strike Price')
    ax2.set_ylabel('Maturity (Years)')
    ax2.set_zlabel('Implied Volatility')
    ax2.set_title(f'{title_prefix} - Put Implied Volatility (3D)')

    plt.tight_layout()
    save_folder = os.path.join(BASE_PLOT_DIR, title_prefix)
    ensure_dir(save_folder)
    plt.savefig(os.path.join(save_folder, f'{title_prefix}_3D_IV_Plot.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_2d_iv_observations(df, title_prefix):
    calls = df[df['Option Type'] == 'CE']
    puts = df[df['Option Type'] == 'PE']

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # IV vs Strike Price
    axes[0, 0].scatter(calls['Strike Price'], calls['Implied_Vol'], alpha=0.5, color='blue')
    axes[0, 0].set_title(f'{title_prefix} - Call IV vs Strike')
    axes[0, 0].set_xlabel('Strike Price')
    axes[0, 0].set_ylabel('Implied Volatility')

    axes[0, 1].scatter(puts['Strike Price'], puts['Implied_Vol'], alpha=0.5, color='red')
    axes[0, 1].set_title(f'{title_prefix} - Put IV vs Strike')
    axes[0, 1].set_xlabel('Strike Price')
    axes[0, 1].set_ylabel('Implied Volatility')

    # IV vs Maturity
    axes[1, 0].scatter(calls['Maturity'], calls['Implied_Vol'], alpha=0.5, color='blue')
    axes[1, 0].set_title(f'{title_prefix} - Call IV vs Maturity')
    axes[1, 0].set_xlabel('Maturity (Years)')
    axes[1, 0].set_ylabel('Implied Volatility')

    axes[1, 1].scatter(puts['Maturity'], puts['Implied_Vol'], alpha=0.5, color='red')
    axes[1, 1].set_title(f'{title_prefix} - Put IV vs Maturity')
    axes[1, 1].set_xlabel('Maturity (Years)')
    axes[1, 1].set_ylabel('Implied Volatility')

    plt.tight_layout()
    save_folder = os.path.join(BASE_PLOT_DIR, title_prefix)
    ensure_dir(save_folder)
    plt.savefig(os.path.join(save_folder, f'{title_prefix}_2D_IV_Plot.png'), dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================
# 4. Execute Script
# ==========================================

print("Generating NIFTY plots...")
plot_3d_iv(df_nifty, "NIFTY")
plot_2d_iv_observations(df_nifty, "NIFTY")

unique_stocks = df_stock['Symbol'].unique()
for symbol in unique_stocks:
    print(f"Generating plots for {symbol}...")
    df_symbol = df_stock[df_stock['Symbol'] == symbol]
    if not df_symbol.empty:
        plot_3d_iv(df_symbol, symbol)
        plot_2d_iv_observations(df_symbol, symbol)

print(f"All Part B plots successfully saved in the '{BASE_PLOT_DIR}' directory!")

Calculating Implied Volatilities (this may take a moment)...
Calculating Implied Volatilities (this may take a moment)...
Generating NIFTY plots...
Generating plots for ITC...
Generating plots for HDFCBANK...
Generating plots for INFY...
Generating plots for RELIANCE...
All Part B plots successfully saved in the 'Output_Plots_Part_B' directory!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import timedelta
import yfinance as yf

# ==========================================
# 1. Historical Data Fetcher & Volatility Calc
# ==========================================

# Path to your local database if you have it saved as CSV
LOCAL_DB_PATH = 'nsedata1.csv'

def fetch_historical_prices(symbol, start_date, end_date):
    """
    Tries to read from your local 'nsedata1' database.
    If not found, it downloads data dynamically via yfinance.
    """
    if os.path.exists(LOCAL_DB_PATH):
        # Assuming your database has columns: 'Date', 'Symbol', 'Close'
        db = pd.read_csv(LOCAL_DB_PATH)
        db['Date'] = pd.to_datetime(db['Date'])
        mask = (db['Symbol'] == symbol) & (db['Date'] >= start_date) & (db['Date'] <= end_date)
        prices = db.loc[mask].sort_values('Date')['Close']
        return prices.values
    else:
        # Fallback to yfinance
        # Map symbols to Yahoo Finance tickers (NIFTY -> ^NSEI, ITC -> ITC.NS)
        yf_ticker = "^NSEI" if symbol.upper() == "NIFTY" else f"{symbol.upper()}.NS"

        try:
            # Fetch data; adding +1 day to end_date to ensure it's inclusive
            data = yf.download(yf_ticker, start=start_date.strftime('%Y-%m-%d'),
                               end=(end_date + timedelta(days=1)).strftime('%Y-%m-%d'),
                               progress=False)
            if not data.empty:
                return data['Close'].values.flatten() # Ensure 1D array
        except Exception as e:
            print(f"Failed to fetch data for {symbol}: {e}")

    return np.array([])

def calculate_historical_volatility(symbol, t0_date, maturity_years):
    """
    Calculates historical volatility by going back a period equal to maturity.
    """
    # Convert maturity back to calendar days
    maturity_days = int(maturity_years * 365)

    # Go back in time for a period equal to maturity
    start_date = t0_date - timedelta(days=maturity_days)

    prices = fetch_historical_prices(symbol, start_date, t0_date)

    if len(prices) < 2:
        return np.nan # Not enough data to calculate standard deviation

    # Calculate continuous log returns
    returns = np.log(prices[1:] / prices[:-1])

    # Standard deviation of returns
    daily_vol = np.std(returns, ddof=1)

    # Annualize (assuming 252 trading days in a year)
    annualized_vol = daily_vol * np.sqrt(252)
    return annualized_vol

# ==========================================
# 2. Compute Historical Volatility (HV)
# ==========================================

print("Computing Historical Volatilities...")

def append_historical_volatility(df, symbol_name):
    # Create an empty column for HV
    df['Historical_Vol'] = np.nan

    # Group by Date and Maturity to avoid redundant API calls/calculations
    unique_maturities = df[['Date', 'Maturity']].drop_duplicates()

    for _, row in unique_maturities.iterrows():
        t0 = row['Date']
        mat = row['Maturity']

        hv = calculate_historical_volatility(symbol_name, t0, mat)

        # Apply the calculated HV to all rows with this specific Date and Maturity
        mask = (df['Date'] == t0) & (df['Maturity'] == mat)
        df.loc[mask, 'Historical_Vol'] = hv

    return df

# Assuming df_nifty and df_stock are carried over from Part B and have 'Implied_Vol'
df_nifty = append_historical_volatility(df_nifty, "NIFTY")

df_stock_list = []
for symbol in df_stock['Symbol'].unique():
    print(f"Fetching HV for {symbol}...")
    df_sym = df_stock[df_stock['Symbol'] == symbol].copy()
    df_sym = append_historical_volatility(df_sym, symbol)
    df_stock_list.append(df_sym)

df_stock_final = pd.concat(df_stock_list)

# ==========================================
# 3. Present Results (Tabular & Graphical)
# ==========================================

OUT_DIR_C = "Output_Part_C"
if not os.path.exists(OUT_DIR_C):
    os.makedirs(OUT_DIR_C)

# --- A. Tabular Form (Save to CSV) ---
# Select relevant columns for the final comparison table
cols_to_keep = ['Date', 'Expiry', 'Strike Price', 'Option Type', 'Maturity', 'Implied_Vol', 'Historical_Vol']

nifty_table = df_nifty[cols_to_keep].copy()
nifty_table.to_csv(f"{OUT_DIR_C}/NIFTY_Volatility_Comparison.csv", index=False)

stock_table = df_stock_final[['Symbol'] + cols_to_keep].copy()
stock_table.to_csv(f"{OUT_DIR_C}/Stock_Volatility_Comparison.csv", index=False)

print(f"Tabular comparison datasets saved in '{OUT_DIR_C}'.")

# --- B. Graphical Form ---
def plot_volatility_comparison(df, symbol, out_dir):
    """
    Plots Implied Volatility (Smile/Smirk) vs Historical Volatility (Flat Line)
    for each unique maturity.
    """
    unique_maturities = df['Maturity'].unique()

    for mat in unique_maturities:
        subset = df[df['Maturity'] == mat]
        if subset.empty or pd.isna(subset['Historical_Vol'].iloc[0]):
            continue

        hv = subset['Historical_Vol'].iloc[0]
        calls = subset[subset['Option Type'] == 'CE']
        puts = subset[subset['Option Type'] == 'PE']

        plt.figure(figsize=(10, 6))

        # Scatter for Implied Volatilities
        plt.scatter(calls['Strike Price'], calls['Implied_Vol'], color='blue', label='Call IV', alpha=0.7)
        plt.scatter(puts['Strike Price'], puts['Implied_Vol'], color='red', label='Put IV', alpha=0.7)

        # Horizontal line for Historical Volatility
        plt.axhline(y=hv, color='green', linestyle='--', linewidth=2, label=f'Historical Volatility (HV = {hv:.4f})')

        mat_days = int(mat * 365)
        plt.title(f'{symbol} - Volatility Comparison (Maturity: {mat_days} days)')
        plt.xlabel('Strike Price')
        plt.ylabel('Volatility (Annualized)')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Save figure
        save_path = os.path.join(out_dir, f"{symbol}_Vol_Comparison_{mat_days}Days.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

# Generate plots
plot_volatility_comparison(df_nifty, "NIFTY", OUT_DIR_C)

for symbol in df_stock_final['Symbol'].unique():
    df_sym = df_stock_final[df_stock_final['Symbol'] == symbol]
    plot_volatility_comparison(df_sym, symbol, OUT_DIR_C)

print(f"All Part C graphical plots successfully saved in the '{OUT_DIR_C}' directory!")

Computing Historical Volatilities...


KeyError: 'Symbol'